##Install Everything

In [9]:
!pip install ultralytics opencv-python-headless deep_sort_realtime
!pip install supervision


##Imports

In [10]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
from google.colab import files
from IPython.display import Video

##Uploaded Video

In [26]:
uploaded = files.upload()

Saving basketball.mp4 to basketball (1).mp4


In [27]:
input_path = list(uploaded.keys())[0]
output_path = 'output.mp4'
print('Input video:', input_path)

Input video: basketball (1).mp4


##Load Models

In [28]:
model = YOLO('yolov8n.pt')
tracker = sv.ByteTrack()

##FULL PROCESSING CODE

In [29]:
cap = cv2.VideoCapture(input_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

frame_count = 0

print('Processing started...')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[detections.class_id == 0]

    tracked = tracker.update_with_detections(detections)

    for box, track_id in zip(tracked.xyxy, tracked.tracker_id):
        if track_id is None:
            continue

        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f'ID {track_id}', (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    out.write(frame)

cap.release()
out.release()

print('Processing complete!')

Processing started...

0: 384x640 2 persons, 160.5ms
Speed: 3.6ms preprocess, 160.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 155.7ms
Speed: 5.9ms preprocess, 155.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 tennis racket, 162.4ms
Speed: 3.2ms preprocess, 162.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 147.3ms
Speed: 2.8ms preprocess, 147.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 214.8ms
Speed: 10.7ms preprocess, 214.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 147.3ms
Speed: 3.3ms preprocess, 147.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 146.2ms
Speed: 3.0ms preprocess, 146.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 238.1ms
Speed: 3.4ms preprocess, 2

##Show Output

In [31]:
Video("final_output.mp4", embed=True)

##Download Output

In [30]:
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>